In [ ]:
from torch.utils.data import DataLoader
import transformers
from datasets import Dataset
from accelerate import Accelerator
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import evaluate

In [ ]:
df = pd.read_csv(r"C:\Users\stasm\SpamClassificator\source.csv")

In [ ]:
df["Category"].value_counts()

In [ ]:
df["lenght"] = df["Message"].str.len()
plt.figure(figsize=(14,6))
sns.set_theme(style = 'whitegrid')
sns.histplot(data = df, x = "lenght", discrete = True, kde = True)

plt.title("Length Distribution")
plt.show()

In [ ]:
tokenizer = transformers.AutoTokenizer.from_pretrained("google-bert/bert-base-cased")

In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["Message"], 
        padding="max_length", 
        truncation=True, 
        max_length=200
    )

In [ ]:
df['label'] = df['Category'].map({'ham' : 0, 'spam' : 1})
hf_dataset = Dataset.from_pandas(df)

In [ ]:
tokenized_dataset = hf_dataset.map(tokenize_function, batched=True)
tokenized_dataset.set_format(type = "torch", columns = ['input_ids', 'attention_mask', 'label'])

In [ ]:
splits = tokenized_dataset.train_test_split(test_size = 0.3, seed = 42)
train_dataset = splits['train']
test_dataset = splits['test']

In [ ]:
accelerator = Accelerator()

In [ ]:
model = transformers.AutoModelForSequenceClassification.from_pretrained(
    "google-bert/bert-base-cased",
    num_labels = 2
)

In [ ]:
trained_args = transformers.TrainingArguments(
    output_dir = "model_args",
    eval_strategy = 'epoch',
    per_device_train_batch_size = 16,
	per_device_eval_batch_size = 16,
	num_train_epochs = 5,
	report_to='none'
)

In [ ]:
metric = evaluate.load("f1")
def compute_metrics(eval_pred):
    logits, label = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [ ]:
trainer = transformers.Trainer(
    model = model,
    args = trained_args,
    train_dataset = train_dataset,
	eval_dataset = test_dataset,
	compute_metrics = compute_metrics
)

In [ ]:
trainer.train()